[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/present_value.ipynb)

# Present value full course: the practice set, in Python

Thirty-six exercises in four tiers, the same as the practice spreadsheet: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (net present value, annuities, loan payments, growing streams, quoted and effective rates),
**tier 3** stretch problems (combine ideas, spot the trap, run a formula backwards) and **tier 4**, the case: Bangalore
Cold Chain (a fictional company; our own data) is offered three ways to be paid for a ten-year storage contract, and
has to decide which is worth the most. Each tier is harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the same present values in plain Python, with
`numpy` for arrays of cash flows and `pandas` for the case's dated cash flows, so you can run them on your own
numbers. And two cells you can move yourself: a discount rate that reprices a stream of cash flows, and a growth
rate that sends a growing stream's value towards the sky.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When a slider appears, move it and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Money in
dollars, to the cent: `1191.02`. Rates as percentages: `7.72` for 7.72 % (typing the % sign is fine, and so is the
rate itself, `0.0772`). The case's values in $ millions, to two decimals: `15.71`. Numbers without thousands
separators: `42123.64`, not `42,123.64`. You can type the arithmetic too: `1000*1.06**3`. For several answers,
separate them with commas, in the order the question lists them (the box's label shows the order). You can also put
your answer between the quotes in a cell's last line, `practice("P1.3", "")`, and run the cell: it checks straight
away.

**The cheat sheet.** Every formula and spreadsheet function from the course on two pages:
[present value cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/present_value_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # cash in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# cash as it is paid CHALK · money moved to today SKY · the rate VIOLET · growth MINT · a shortcut that misleads CORAL ·
# the one thing to look at GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="our own example · computed"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "exp": math.exp, "ln": math.log, "log": math.log}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets and sqrt(…), exp(…), ln(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"\b(dollars?|years?|months?|payments?|millions?|percent)\b|[$%]"

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    t = re.sub(UNITS, "", s, flags=re.I).strip()
    try: return calc(t)
    except Exception: return " ".join(s.lower().split())

def right(got, want, tol, pct=False):
    """Within the tolerance; a rate asked for in percent may also be typed as the rate itself (0.0772 for 7.72 %)."""
    if isinstance(got, str): return False
    tol = max(tol, 1e-7)
    return abs(got - want) <= tol or (pct and abs(got * 100 - want) <= tol)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the same questions and answers as the spreadsheet) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'The goal of the firm',
     "q": 'A firm builds a cold store, takes a bank loan and sells new shares. Which of the three is a real asset the firm buys (type 1, 2 or 3)?',
     "parts": ['the real asset (1, 2 or 3)'],
     "answer": [1], "tol": [0], "pct": [], "function": '—',
     "working": 'The cold store is a real asset (it produces the business); the loan and the shares are financial assets the firm sells to pay for it: claims on its future cash'},
    {"id": 'P1.2', "tier": 1, "group": 'The goal of the firm',
     "q": 'A project costs $2,000 today and pays $2,150 in one year. What is its rate of return, in percent?',
     "parts": ['rate of return (%)'],
     "answer": [7.5], "tol": [0.005], "pct": [0], "function": '—',
     "working": '2,150/2,000 − 1 = 0.075 = 7.5 %'},
    {"id": 'P1.3', "tier": 1, "group": 'Future and present value',
     "q": 'What is $3,000 worth after 4 years at 5 % a year, compounded yearly?',
     "parts": ['future value ($)'],
     "answer": [3646.5187500000006], "tol": [0.005], "pct": [], "function": 'FV',
     "working": '3,000 × 1.05⁴ = 3,000 × 1.215506 = 3,646.52'},
    {"id": 'P1.4', "tier": 1, "group": 'Future and present value',
     "q": 'What is the 3-year discount factor at 7 %? Give four decimals.',
     "parts": ['discount factor (four decimals)'],
     "answer": [0.8162978768908519], "tol": [5e-05], "pct": [], "function": '—',
     "working": 'DF₃ = 1/1.07³ = 1/1.225043 = 0.8163'},
    {"id": 'P1.5', "tier": 1, "group": 'Future and present value',
     "q": 'What is $5,000 received in 2 years worth today at 10 %?',
     "parts": ['present value ($)'],
     "answer": [4132.231404958677], "tol": [0.005], "pct": [], "function": 'PV',
     "working": '5,000/1.1² = 5,000/1.21 = 4,132.23'},
    {"id": 'P1.6', "tier": 1, "group": 'Perpetuities and annuities',
     "q": 'A fund must pay $12,000 a year forever, the first payment in a year. At 4 %, what must it hold today?',
     "parts": ['what the fund must hold ($)'],
     "answer": [300000.0], "tol": [0.005], "pct": [], "function": '—',
     "working": 'PV = C/r = 12,000/0.04 = 300,000'},
    {"id": 'P1.7', "tier": 1, "group": 'Perpetuities and annuities',
     "q": 'What is the 4-year annuity factor at 8 %? Give four decimals.',
     "parts": ['annuity factor (four decimals)'],
     "answer": [3.3121268400443338], "tol": [5e-05], "pct": [], "function": 'PV',
     "working": '1/0.08 − 1/(0.08 × 1.08⁴) = 12.5 − 9.1879 = 3.3121'},
    {"id": 'P1.8', "tier": 1, "group": 'Growing perpetuities and annuities',
     "q": 'A payment of $5,000 next year will grow 2 % a year forever. At 7 %, what is it worth today?',
     "parts": ['present value ($)'],
     "answer": [100000.0], "tol": [0.005], "pct": [], "function": '—',
     "working": 'C₁/(r − g) = 5,000/0.05 = 100,000'},
    {"id": 'P1.9', "tier": 1, "group": 'How interest is quoted',
     "q": 'A loan quotes 6 % a year compounded monthly. What is the effective annual rate, in percent?',
     "parts": ['effective annual rate (%)'],
     "answer": [6.167781186449828], "tol": [0.005], "pct": [0], "function": 'EFFECT',
     "working": '(1 + 0.06/12)¹² − 1 = 1.005¹² − 1 = 6.1678 %'},
    {"id": 'P1.10', "tier": 1, "group": 'Nominal and real',
     "q": 'A deposit pays 5 % and prices rise 2 %. What is the real rate, in percent (exact, not the shortcut)?',
     "parts": ['real rate (%)'],
     "answer": [2.941176470588247], "tol": [0.005], "pct": [0], "function": '—',
     "working": '1.05/1.02 − 1 = 2.9412 % (the shortcut 5 − 2 = 3 % is a little high)'},
    {"id": 'P2.1', "tier": 2, "group": 'Future and present value',
     "q": 'A machine costs $1,500 today and returns $400, $400 and $900 at the ends of years 1, 2 and 3. At 9 %, what is its NPV?',
     "parts": ['NPV ($)'],
     "answer": [-101.39039357419824], "tol": [0.005], "pct": [], "function": 'NPV',
     "working": 'PVs 366.97 + 336.67 + 694.97 = 1398.61; NPV = 1398.61 − 1,500 = -101.39. In a spreadsheet: =-1500 + NPV(9%, 400, 400, 900)'},
    {"id": 'P2.2', "tier": 2, "group": 'Future and present value',
     "q": 'Same machine: what does =NPV(9%, -1500, 400, 400, 900) give, and why is it wrong?',
     "parts": ['what the one-cell NPV returns ($)'],
     "answer": [-93.01870970109928], "tol": [0.005], "pct": [], "function": 'NPV',
     "working": "-93.02 = the right NPV ÷ 1.09: the function discounts its first value one period, so today's cost is pushed a year into the future. Add C₀ outside the function"},
    {"id": 'P2.3', "tier": 2, "group": 'Future and present value',
     "q": 'How many years does money take to double at 8 % a year? Two decimals.',
     "parts": ['years to double (two decimals)'],
     "answer": [9.006468342000588], "tol": [0.005], "pct": [], "function": 'NPER, LN',
     "working": '1.08ᵗ = 2 → t = ln 2/ln 1.08 = 9.01 years. In a spreadsheet: =NPER(8%, 0, -1, 2, 0)'},
    {"id": 'P2.4', "tier": 2, "group": 'Perpetuities and annuities',
     "q": 'A five-year lease pays $18,000 at the end of each year. At 6 %, what is it worth today? And if each payment came at the start of the year?',
     "parts": ['paid at the end of each year ($)', 'paid at the start of each year ($)'],
     "answer": [75822.54814018292, 80371.9010285939], "tol": [0.005, 0.005], "pct": [], "function": 'PV',
     "working": '18,000 × 4.2124 = 75,822.55; at the start: × 1.06 = 80,371.90. =PV(6%, 5, -18000, 0, 0) and =PV(6%, 5, -18000, 0, 1)'},
    {"id": 'P2.5', "tier": 2, "group": 'Perpetuities and annuities',
     "q": 'A $40,000 equipment loan at 7 % a year is repaid in 6 equal end-of-year payments. What is the payment?',
     "parts": ['the payment ($)'],
     "answer": [8391.83199033312], "tol": [0.005], "pct": [], "function": 'PMT',
     "working": '40,000/4.7665 = 8,391.83. =PMT(7%, 6, 40000, 0, 0) gives it with a minus sign (money paid out)'},
    {"id": 'P2.6', "tier": 2, "group": 'Perpetuities and annuities',
     "q": 'You save $6,000 at the end of each year for 8 years at 5 %. How much do you have at the end?',
     "parts": ['the balance at the end ($)'],
     "answer": [57294.65325468757], "tol": [0.005], "pct": [], "function": 'FV',
     "working": 'PV 38,779.28 × 1.05⁸ = 57,294.65. =FV(5%, 8, -6000, 0, 0)'},
    {"id": 'P2.7', "tier": 2, "group": 'Growing perpetuities and annuities',
     "q": 'A licence pays $30,000 next year, growing 3 % a year for 12 years. At 8 %, what is it worth today?',
     "parts": ['present value ($)'],
     "answer": [260286.44117644292], "tol": [0.005], "pct": [], "function": '—',
     "working": '30,000/0.05 × (1 − (1.03/1.08)¹²) = 600,000 × 0.433811 = 260,286.44'},
    {"id": 'P2.8', "tier": 2, "group": 'How interest is quoted',
     "q": "Bank A quotes 7.8 % compounded monthly; bank B 7.9 % compounded yearly. For a deposit, which pays more? Give A's effective rate in percent.",
     "parts": ["bank A's effective rate (%)"],
     "answer": [8.084981036551554], "tol": [0.005], "pct": [0], "function": 'EFFECT',
     "working": 'A: (1 + 0.078/12)¹² − 1 = 8.0850 % > 7.9 %: A pays more. =EFFECT(7.8%, 12)'},
    {"id": 'P2.9', "tier": 2, "group": 'How interest is quoted',
     "q": '$2,000 grows at 5 % a year compounded continuously for 6 years. How much is there at the end?',
     "parts": ['the amount at the end ($)'],
     "answer": [2699.717615152006], "tol": [0.005], "pct": [], "function": 'EXP',
     "working": '2,000 × e^(0.05 × 6) = 2,000 × e^0.3 = 2,699.72. =2000*EXP(0.05*6)'},
    {"id": 'P2.10', "tier": 2, "group": 'Nominal and real',
     "q": "A rent of $24,000 is fixed for 5 years. With 2.5 % inflation, what is the last payment worth in today's dollars?",
     "parts": ["the last payment in today's dollars ($)"],
     "answer": [21212.502902628417], "tol": [0.005], "pct": [], "function": '—',
     "working": '24,000/1.025⁵ = 21,212.50'},
    {"id": 'P3.1', "tier": 3, "group": 'The goal of the firm',
     "q": 'A project costs $80,000 and pays $86,000 in a year. Shareholders can earn 6 % on safe deposits and 10 % on investments as risky as this one. Is it worth doing? Give its NPV at the right rate.',
     "parts": ['NPV at the right rate ($)'],
     "answer": [-1818.1818181818235], "tol": [0.005], "pct": [], "function": '—',
     "working": 'Discount at the return given up for the same risk, 10 %: 86,000/1.1 − 80,000 = -1,818.18: no. At the safe 6 % it would look good (1,132.08), but that is the wrong yardstick'},
    {"id": 'P3.2', "tier": 3, "group": 'Perpetuities and annuities',
     "q": 'A supplier sells a $9,000 machine for 4 payments of $2,600, the first in a year. What rate is the plan charging, in percent?',
     "parts": ['the rate the plan charges (%)'],
     "answer": [6.044994808143249], "tol": [0.005], "pct": [0], "function": 'RATE',
     "working": 'Solve 2,600 × annuity factor(r, 4) = 9,000: r ≈ 6.04 %. =RATE(4, -2600, 9000, 0, 0)'},
    {"id": 'P3.3', "tier": 3, "group": 'Perpetuities and annuities',
     "q": 'A $30,000 loan at 6 % is repaid at $5,000 a year. How many payments does it take (two decimals)?',
     "parts": ['number of payments (two decimals)'],
     "answer": [7.659094995891753], "tol": [0.005], "pct": [], "function": 'NPER',
     "working": 'From 30,000 = 5,000 × (1 − 1.06⁻ⁿ)/0.06: 1.06⁻ⁿ = 0.64, n = ln(1/0.64)/ln 1.06 = 7.66. =NPER(6%, -5000, 30000, 0, 0)'},
    {"id": 'P3.4', "tier": 3, "group": 'Growing perpetuities and annuities',
     "q": 'A stream priced at $1,500,000 pays $60,000 next year and grows forever. At 7 %, what growth rate does the price assume, in percent?',
     "parts": ['the growth rate the price assumes (%)'],
     "answer": [3.0000000000000004], "tol": [0.005], "pct": [0], "function": '—',
     "working": 'PV = C₁/(r − g) → g = r − C₁/PV = 7 % − 4 % = 3 %'},
    {"id": 'P3.5', "tier": 3, "group": 'Growing perpetuities and annuities',
     "q": 'At 6 %, a payment of $10,000 next year grows at g forever. Value it at g = 0, 3 % and 5 %, and say what happens as g approaches 6 %.',
     "parts": ['value at g = 0 ($)', 'value at g = 3 % ($)', 'value at g = 5 % ($)'],
     "answer": [166666.6666666667, 333333.3333333334, 1000000.0000000005], "tol": [0.005, 0.005, 0.005], "pct": [], "function": '—',
     "working": '166,666.67 · 333,333.33 · 1,000,000: as g → r the denominator r − g → 0 and the value has no limit; the formula needs g < r'},
    {"id": 'P3.6', "tier": 3, "group": 'How interest is quoted',
     "q": "A firm's cost of capital is 9 % effective a year. What monthly rate should it use for monthly cash flows, in percent (four decimals)? What is wrong with 9/12 = 0.75 %?",
     "parts": ['the monthly rate (%, four decimals)'],
     "answer": [0.7207323316136716], "tol": [5e-05], "pct": [0], "function": '—',
     "working": '(1.09)^(1/12) − 1 = 0.7207 %. 0.75 % a month compounds to 9.38 % a year, not 9 %'},
    {"id": 'P3.7', "tier": 3, "group": 'Nominal and real',
     "q": "A three-year rent pays $10,000 next year, rising with 3 % inflation. At a 9 % nominal rate, what is it worth? Then value the same rent in today's dollars at the real rate, and say what goes wrong if you discount the rising rent at the real rate.",
     "parts": ['the rising rent at the nominal rate ($)', "today's dollars at the real rate ($)", 'the mix-up: the rising rent at the real rate ($)'],
     "answer": [26035.7103972189, 26035.710397218903, 27598.52871480098], "tol": [0.005, 0.005, 0.005], "pct": [], "function": '—',
     "working": 'Nominal: 26,035.71. Real: 9,708.74 a year at 5.825 % → 26,035.71, the same. Mixed (nominal flows, real rate): 27,598.53, too high: inflation counted in the cash but not in the rate'},
    {"id": 'P3.8', "tier": 3, "group": 'Nominal and real',
     "q": 'You want a 4 % real return and expect 3 % inflation. What nominal rate must you earn, in percent? How far off is the shortcut 4 + 3?',
     "parts": ['the nominal rate (%)'],
     "answer": [7.120000000000015], "tol": [0.005], "pct": [0], "function": '—',
     "working": '(1.04)(1.03) − 1 = 7.12 %; the shortcut misses the 0.04 × 0.03 = 0.12 point cross term'},
    {"id": 'P4.1', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "Add up each offer's payments with no discounting (in $ millions). Which looks best, and why is the comparison meaningless?",
     "parts": ['offer A, added up ($ millions)', 'offer B, added up ($ millions)', 'offer C, added up ($ millions)'],
     "answer": [15.0, 22.927758622941464, 20.4], "tol": [0.005, 0.005, 0.005], "pct": [], "function": 'SUM',
     "working": 'A 15.00 · B 22.93 · C 20.40: B, then C. But the sums add dollars from different dates; only money at the same date can be added'},
    {"id": 'P4.2', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'Value offer B at 9 % twice: as if each payment came at the end of the year, and as it really comes, at the start ($ millions).',
     "parts": ['B as if paid at the end of each year ($ millions)', 'B paid at the start, as it is ($ millions)'],
     "answer": [14.410506593363879, 15.70745218676663], "tol": [0.005, 0.005], "pct": [], "function": 'SUMPRODUCT',
     "working": 'End of year: 2/(0.09 − 0.03) × (1 − (1.03/1.09)¹⁰) = 14.4105; at the start: × 1.09 = 15.7075. The timing alone moves B from below A to above it'},
    {"id": 'P4.3', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "Value B again in real terms: $2 million a year in today's dollars, at the real rate. Then show the mixed-up version (the same flat $2 million at 9 %).",
     "parts": ['B in real terms at the real rate ($ millions)', 'the mix-up: a flat $2 million at 9 % ($ millions)'],
     "answer": [15.707452186766632, 13.990493788526647], "tol": [0.005, 0.005], "pct": [], "function": 'PV',
     "working": 'Real rate 5.825 %: 15.7075, the same as nominal. Mixed: 13.9905, too low: inflation left out of the cash but kept in the rate'},
    {"id": 'P4.4', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'Value offer C ($ millions) with the right monthly rate, and with 9 %/12. In a spreadsheet: =PV(rate, 120, -170000, 0, 0).',
     "parts": ['C at the right monthly rate ($ millions)', 'C at 9 %/12 ($ millions)'],
     "answer": [13.623665613542249, 13.420087753730105], "tol": [0.005, 0.005], "pct": [], "function": 'PV',
     "working": 'Monthly rate 1.09^(1/12) − 1 = 0.7207 %: 13.6237; with 0.75 %: 13.4201'},
    {"id": 'P4.5', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": "If the grocer's payments are riskier than the company's usual business, B deserves a higher rate. At what rate (percent) is B worth exactly the $15 million of offer A?",
     "parts": ['the rate at which B is worth $15 million (%)'],
     "answer": [10.266709811693303], "tol": [0.005], "pct": [0], "function": 'RATE (or goal seek)',
     "working": 'Solve the growing annuity due = 15 million: r ≈ 10.27 %. Above that, A wins'},
    {"id": 'P4.6', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'B rises with inflation. At what yearly growth rate (percent) would B be worth exactly $15 million at 9 %?',
     "parts": ['the growth at which B is worth $15 million (%)'],
     "answer": [1.8167679000559398], "tol": [0.005], "pct": [0], "function": 'goal seek',
     "working": 'g ≈ 1.82 %: inflation would have to stay below that for A to win'},
    {"id": 'P4.7', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'Suppose the grocer offered to renew B on the same terms forever. What would that be worth ($ millions), and why is it only a ceiling?',
     "parts": ['B renewed forever ($ millions)'],
     "answer": [36.333333333333336], "tol": [0.005], "pct": [], "function": '—',
     "working": 'A growing perpetuity due: 2 × 1.09/(0.09 − 0.03) = 36.3333. No grocer guarantees that; it is the most the stream could ever be worth'},
    {"id": 'P4.8', "tier": 4, "group": 'Case study · Bangalore Cold Chain',
     "q": 'Write the recommendation in two sentences, with the numbers that anchor it.',
     "parts": ["the anchor: B's edge over A ($ millions)"],
     "answer": [0.70745218676663], "tol": [0.005], "pct": [], "function": '—',
     "working": "Take offer B: worth 15.71 million at 9 %, 0.71 million more than A and 2.08 more than C. It stays ahead while the grocer's risk deserves less than 10.3 % and inflation stays above 1.8 %"},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch", 4: "Case"}
assert len(EX) == 36 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, t, i in q["pct"]) for i, (g, w, t) in enumerate(zip(got, q["answer"], q["tol"]))]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            if q["function"] not in ("—", ""): print(textwrap.fill(f"In a spreadsheet: {q['function']}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers practice.py gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(float(g), w, t) for g, w, t in zip(got, answers(k), Q[k]["tol"]))

# the checker itself: a right answer passes, a wrong one doesn't; a rate may be typed as a percentage or as itself
assert right(read_part("3000*1.05**4"), answers("P1.3")[0], 0.005) and not right(read_part("3646"), answers("P1.3")[0], 0.005)
assert right(read_part("7.5%"), 7.5, 0.005, True) and right(read_part("0.075"), 7.5, 0.005, True)
assert not right(read_part("0.075"), 7.5, 0.005) and not right(read_part("seven"), 7.5, 0.005, True)

## Tier 1 · Warm-up

One idea, one step, small numbers: a real asset among financial ones, a rate of return, a future value, a discount
factor, a present value, a perpetuity, an annuity factor, a growing perpetuity, an effective annual rate, and a real
rate.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Python has no built-in PV or PMT, and doesn't need one: each formula from the course is one line. The cell below
writes them once (the comment beside each names the spreadsheet function that does the same job), then computes
every warm-up answer and checks it against the practice set.

In [ ]:
def fv(amount, r, t): return amount * (1 + r) ** t                 # FV(r, t, 0, -amount, 0)
def pv(amount, r, t): return amount / (1 + r) ** t                 # PV(r, t, 0, -amount, 0)
def perpetuity(c, r): return c / r                                 # the first payment in one period
def annuity_factor(r, t): return 1 / r - 1 / (r * (1 + r) ** t)   # two perpetuities, one starting t periods later
def annuity(c, r, t, due=False):                                   # PV(r, t, -c, 0, 0); due=True is type 1
    return c * annuity_factor(r, t) * ((1 + r) if due else 1)
def pmt(loan, r, t): return loan / annuity_factor(r, t)            # −PMT(r, t, loan, 0, 0)
def growing_perpetuity(c1, r, g): return c1 / (r - g)              # needs g < r
def growing_annuity(c1, r, g, t): return c1 / (r - g) * (1 - ((1 + g) / (1 + r)) ** t)
def ear(quoted, m): return (1 + quoted / m) ** m - 1               # EFFECT(quoted, m)
def quoted_rate(effective, m): return m * ((1 + effective) ** (1 / m) - 1)   # NOMINAL(effective, m)
def real_rate(nominal, inflation): return (1 + nominal) / (1 + inflation) - 1

ret = 2150 / 2000 - 1                                              # P1.2: a rate of return
fv4 = fv(3000, 0.05, 4)                                            # P1.3
df3 = 1 / 1.07 ** 3                                                # P1.4: a discount factor
pv2 = pv(5000, 0.10, 2)                                            # P1.5
fund = perpetuity(12000, 0.04)                                     # P1.6
af = annuity_factor(0.08, 4)                                       # P1.7
gp = growing_perpetuity(5000, 0.07, 0.02)                          # P1.8
e12 = ear(0.06, 12)                                                # P1.9
real = real_rate(0.05, 0.02)                                       # P1.10

print(f"return {ret:.1%}   FV {fv4:,.2f}   discount factor {df3:.4f}   PV {pv2:,.2f}   fund {fund:,.0f}")
print(f"annuity factor {af:.4f}   growing perpetuity {gp:,.0f}   effective rate {e12:.4%}   real rate {real:.4%}")
for k, v in (("P1.1", [1]), ("P1.2", [100 * ret]), ("P1.3", [fv4]), ("P1.4", [df3]), ("P1.5", [pv2]), ("P1.6", [fund]),
             ("P1.7", [af]), ("P1.8", [gp]), ("P1.9", [100 * e12]), ("P1.10", [100 * real])):
    assert agrees(k, v), k

### The machine's stream, in three lines

The course's machine costs $1,000 today and returns $300, $400 and $500 at the ends of years 1, 2 and 3. At 8 %, each
cash flow is divided by 1.08 once for every year it is away, and the results are added: a net present value of
$17.63, the number the video worked out by hand.

In [ ]:
flows = np.array([300, 400, 500])
years = np.array([1, 2, 3])
npv = -1000 + np.sum(flows / 1.08 ** years)
print(f"NPV = {npv:.2f}")

In [ ]:
# The same in one function, for any stream: flows[k] arrives at time first + k.
def present_value(rate, flows, first=0):
    t = first + np.arange(len(flows))
    return float(np.sum(np.asarray(flows, float) / (1 + rate) ** t))

machine = [-1000, 300, 400, 500]
right_npv = present_value(0.08, machine)                 # the cost is today's, at time 0
wrong_npv = present_value(0.08, machine, first=1)        # a spreadsheet's NPV with the cost inside: everything a year late
print(f"first flow today: {right_npv:.2f}   pushed a year late: {wrong_npv:.2f} = {right_npv:.2f} / 1.08")
assert round(float(npv), 2) == round(right_npv, 2) == 17.63 and round(wrong_npv, 2) == 16.32         # the video's numbers
assert round(present_value(0.08, flows, first=1), 2) == 1017.63 and abs(wrong_npv - right_npv / 1.08) < 1e-9

### Slide the rate

The pale bars are the machine's cash flows as they are paid; the SKY bars are the same cash flows moved to today.
At 0 % nothing shrinks and the NPV is the plain sum, $200. At 8 % it is $17.63. Keep sliding: a little under 9 %
the NPV crosses zero, and above that the machine destroys value. The cash flows never changed; only the return
given up elsewhere did.

In [ ]:
def npv_picture(rate_pct=8.0):
    r = rate_pct / 100
    cash = np.array(machine, float); t = np.arange(len(cash)); today = cash / (1 + r) ** t
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.09, right=0.98, top=0.84, bottom=0.14)
    ax.bar(t - 0.19, cash, width=0.36, color=CHALK, alpha=0.45, label="cash flow, as paid")
    ax.bar(t + 0.19, today, width=0.36, color=SKY, label="its value today")
    ax.axhline(0, color=DIM, lw=1)
    ax.set_xticks(t); ax.set_xticklabels(["today", "year 1", "year 2", "year 3"]); ax.set_ylim(-1150, 650)
    ax.set_ylabel("dollars"); ax.legend(loc="lower right", frameon=False)
    total = today.sum()
    fig.suptitle(f"rate {rate_pct:g} %: NPV = {total:,.2f}".replace("-", "−"), x=0.09, ha="left", fontsize=15,
                 color=GOLD if total >= 0 else CORAL)
    footnote(fig); plt.show()

assert round(present_value(0.0, machine), 6) == 200

In [ ]:
interact(npv_picture, rate_pct=FloatSlider(value=8, min=0, max=20, step=0.5, description="rate (%)"));

## Tier 2 · Core

The module's standard problems, several steps each: a net present value and the spreadsheet trap beside it, a
doubling time, an annuity paid at the end and at the start of each year, a loan payment, savings that build up, a
growing annuity, two banks' quoted rates, continuous compounding, and a fixed rent in today's dollars.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

**A stream of cash flows** is an array divided by the discount factors and summed: `present_value(rate, flows)`
from above. **Annuities** use `annuity(c, r, t)`, with `due=True` when each payment comes at the start of its
period; a loan payment is the loan divided by the annuity factor; a future value is the present value carried
forward with `(1 + r) ** t`. **Continuous compounding** is `math.exp(r * t)`. Each line below computes one core
answer and checks it against the practice set.

In [ ]:
p21 = present_value(0.09, [-1500, 400, 400, 900])                  # P2.1: the cost is at time 0
p22 = present_value(0.09, [-1500, 400, 400, 900], first=1)         # P2.2: the cost pushed a year late
double = math.log(2) / math.log(1.08)                              # P2.3: 1.08^t = 2
lease_end, lease_start = annuity(18000, 0.06, 5), annuity(18000, 0.06, 5, due=True)    # P2.4
payment = pmt(40000, 0.07, 6)                                      # P2.5
saved = annuity(6000, 0.05, 8) * 1.05 ** 8                         # P2.6: value the savings today, then carry forward
licence = growing_annuity(30000, 0.08, 0.03, 12)                   # P2.7
bank_a = ear(0.078, 12)                                            # P2.8: against bank B's 7.9 %
cont = 2000 * math.exp(0.05 * 6)                                   # P2.9
rent_real = 24000 / 1.025 ** 5                                     # P2.10

print(f"NPV {p21:,.2f}   the one-cell NPV {p22:,.2f}   doubling in {double:.2f} years")
print(f"lease {lease_end:,.2f} (end of year) or {lease_start:,.2f} (start)   loan payment {payment:,.2f}   "
      f"savings {saved:,.2f}")
print(f"licence {licence:,.2f}   bank A {bank_a:.4%} against 7.9 %   continuous {cont:,.2f}   "
      f"the last rent in today's dollars {rent_real:,.2f}")
for k, v in (("P2.1", [p21]), ("P2.2", [p22]), ("P2.3", [double]), ("P2.4", [lease_end, lease_start]),
             ("P2.5", [payment]), ("P2.6", [saved]), ("P2.7", [licence]), ("P2.8", [100 * bank_a]), ("P2.9", [cont]),
             ("P2.10", [rent_real])):
    assert agrees(k, v), k

A loan paid off, year by year. The course's loan is $30,000 at 6 %, repaid in five equal payments of $7,121.89. Each
payment covers the year's interest first; the rest pays down the balance, and the fifth payment brings it to
exactly zero.

In [ ]:
loan, rate, n_years = 30000, 0.06, 5
pay = pmt(loan, rate, n_years)
rows, balance = [], loan
for year in range(1, n_years + 1):
    interest = balance * rate
    rows.append({"year": year, "opening balance": balance, "interest": interest, "payment": pay,
                 "closing balance": balance + interest - pay})
    balance = balance + interest - pay
schedule = pd.DataFrame(rows).set_index("year")
print(schedule.round(2).to_string())
assert round(pay, 2) == 7121.89 and round(schedule["interest"].iloc[0], 6) == 1800 and abs(balance) < 1e-6

### Slide the growth rate

A payment of $40,000 next year that grows at g forever, valued at 6 %: C₁/(r − g). With no growth it is worth
$666,667; at 2 % growth, $1,000,000; at 4 %, $2,000,000. Slide g towards 6 % and the value climbs without limit: the
formula only holds while the growth rate stays below the discount rate (the VIOLET line).

In [ ]:
def growth_picture(g_pct=2.0):
    r, c1 = 0.06, 40000
    gs = np.linspace(0, 0.0595, 300)
    value = growing_perpetuity(c1, r, g_pct / 100)
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
    ax.plot(100 * gs, c1 / (r - gs) / 1e6, color=MINT)
    ax.axvline(100 * r, color=VIOLET, ls="--", lw=1.8)
    ax.text(100 * r - 0.06, 0.5, "g = r: no value fits", color=VIOLET, ha="right", va="bottom", fontsize=12)
    ax.scatter([g_pct], [min(value / 1e6, 10)], s=110, color=GOLD, zorder=4)
    ax.set_xlim(0, 6.3); ax.set_ylim(0, 10); ax.set_xlabel("growth rate g (% a year)")
    ax.set_ylabel("value today ($ millions)")
    fig.suptitle(f"g = {g_pct:g} %: 40,000 / ({100 * r:g} % − {g_pct:g} %) = {value:,.0f}", x=0.08, ha="left",
                 fontsize=15, color=GOLD)
    footnote(fig); plt.show()

assert [round(growing_perpetuity(40000, 0.06, g), 2) for g in (0, 0.02, 0.04)] == [666666.67, 1000000, 2000000]

In [ ]:
interact(growth_picture, g_pct=FloatSlider(value=2, min=0, max=5.9, step=0.1, description="g (%)"));

## Tier 3 · Stretch

Combine ideas, choose the yardstick, spot the trap, run a formula backwards: the rate that matches a project's risk,
the rate hidden in a payment plan, how long a loan takes, the growth a price assumes, what happens as g approaches
r, a monthly rate from a yearly one, a rent valued in nominal and in real terms, and the nominal rate a real target
needs.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

**Running a formula backwards.** The number of payments comes out with logarithms. The rate has no formula: a
spreadsheet's RATE tries values until the present value matches, and `solve` below does the same by halving an
interval. **Matching the rate to the cash flows**: monthly cash flows need the monthly rate, `1.09 ** (1/12) - 1`,
and cash flows in today's dollars need the real rate. Each line computes one stretch answer and checks it.

In [ ]:
def solve(f, lo, hi):
    """The x between lo and hi where f(x) = 0, by halving the interval (f must change sign between them)."""
    f_lo = f(lo)
    for _ in range(200):
        mid = (lo + hi) / 2
        if (f(mid) > 0) == (f_lo > 0): lo, f_lo = mid, f(mid)
        else: hi = mid
    return (lo + hi) / 2

p31 = pv(86000, 0.10, 1) - 80000                                   # P3.1: the rate for this risk, not the safe 6 %
plan_rate = solve(lambda r: annuity(2600, r, 4) - 9000, 1e-9, 1)   # P3.2: RATE(4, -2600, 9000, 0, 0)
n_pay = math.log(1 / (1 - 30000 * 0.06 / 5000)) / math.log(1.06)   # P3.3: NPER(6%, -5000, 30000, 0, 0)
implied_g = 0.07 - 60000 / 1500000                                 # P3.4: g = r − C₁/PV
values = [growing_perpetuity(10000, 0.06, g) for g in (0, 0.03, 0.05)]   # P3.5
monthly = 1.09 ** (1 / 12) - 1                                     # P3.6: not 0.09/12
rent = 10000 * 1.03 ** np.arange(3)                                # P3.7: the rent as paid, years 1 to 3
rent_today = rent / 1.03 ** np.arange(1, 4)                        #        the same rent in today's dollars
real9 = real_rate(0.09, 0.03)
rent_nominal = present_value(0.09, rent, first=1)
rent_real_terms = present_value(real9, rent_today, first=1)
rent_mixed = present_value(real9, rent, first=1)                   #        the trap: rising rent at the real rate
need = 1.04 * 1.03 - 1                                             # P3.8

print(f"NPV at 10 %: {p31:,.2f} (at the safe 6 % it would look like {pv(86000, 0.06, 1) - 80000:,.2f})")
print(f"the plan charges {plan_rate:.4%}   payments needed {n_pay:.2f}   growth the price assumes {implied_g:.2%}")
print("growing perpetuity at g = 0, 3 %, 5 %:", ", ".join(f"{v:,.2f}" for v in values))
print(f"monthly rate {monthly:.4%} (0.75 % a month would compound to {ear(0.09, 12):.2%} a year)")
print(f"rent: nominal {rent_nominal:,.2f} = real {rent_real_terms:,.2f}; mixed up {rent_mixed:,.2f}   "
      f"nominal rate needed {need:.2%}")
for k, v in (("P3.1", [p31]), ("P3.2", [100 * plan_rate]), ("P3.3", [n_pay]), ("P3.4", [100 * implied_g]),
             ("P3.5", values), ("P3.6", [100 * monthly]), ("P3.7", [rent_nominal, rent_real_terms, rent_mixed]),
             ("P3.8", [100 * need])):
    assert agrees(k, v), k
assert abs(rent_nominal - rent_real_terms) < 1e-6 and rent_mixed > rent_nominal

## Tier 4 · The case: Bangalore Cold Chain's three offers

Bangalore Cold Chain runs refrigerated warehouses (a fictional company; our own data). A grocery chain offers it a
ten-year contract to run its frozen storage, and three ways to be paid:

| offer | what it pays |
|---|---|
| A | $15 million today |
| B | $2 million at the start of each year for ten years, rising 3 % a year with expected inflation |
| C | $170,000 at the end of each month for ten years |

The company's opportunity cost of capital is 9 % a year (effective): what its investors could earn on investments of the
same risk. **Which offer is worth the most?** The next cell builds the three offers as dated cash flows, one row a
month (the same table as the spreadsheet's Case study tab), and checks a few of its numbers.

In [ ]:
COST_OF_CAPITAL, INFLATION = 0.09, 0.03
OFFER_A, OFFER_B, OFFER_C = 15_000_000, 2_000_000, 170_000        # today · the first yearly payment · each month

month = np.arange(121)
offers = pd.DataFrame({"month": month, "year": month / 12})
offers["A"] = np.where(month == 0, OFFER_A, 0.0)
offers["B"] = np.where((month % 12 == 0) & (month < 120), OFFER_B * (1 + INFLATION) ** (month // 12), 0.0)
offers["C"] = np.where(month >= 1, OFFER_C, 0.0)
offers["factor"] = 1 / (1 + COST_OF_CAPITAL) ** offers["year"]    # the discount factor for each date

# The case's numbers as the course computed them, to check this table against.
# @STATS
CASE_STATS = {'sum_A': 15000000.0,
 'sum_B': 22927758.622941464,
 'sum_C': 20400000.0,
 'B_ord': 14410506.59336388,
 'B': 15707452.18676663,
 'B_brute': 15707452.186766624,
 'real': 0.058252427184465994,
 'B_real': 15707452.186766632,
 'B_mixed': 13990493.788526647,
 'm': 0.007207323316136716,
 'm_wrong': 0.0075,
 'C': 13623665.61354225,
 'C_wrong': 13420087.753730105,
 'C_brute': 13623665.613542264,
 'edge': 707452.18676663,
 'B_g2': 15106888.741687225,
 'B_ceiling': 36333333.333333336,
 'B_even_r': 0.10266709811693303,
 'B_even_g': 0.0181676790005594}
# @END STATS
S = CASE_STATS
assert len(offers) == 121 and (offers["B"] > 0).sum() == 10 and (offers["C"] > 0).sum() == 120
assert abs(offers["B"].sum() - S["sum_B"]) < 1e-6 and offers["C"].sum() == S["sum_C"] and offers["A"].sum() == S["sum_A"]
print(f"{len(offers)} rows (months 0 to 120), {offers.shape[1]} columns: the same cash flows as the course's case.")
offers[offers["month"] % 12 == 0].head(4)

### How you'd do this in Python

With the offers in a DataFrame, valuing one is a column times the discount factors, summed. The formulas from the
course give the same numbers in one line each. The cell below reproduces the video's case: the sum that means
nothing, then offer B at the wrong timing and the right one, in nominal and in real terms, then offer C at the
right monthly rate and the wrong one.

In [ ]:
million = 1e6
added = {k: offers[k].sum() for k in "ABC"}                                               # no discounting
dated = {k: (offers[k] * offers["factor"]).sum() for k in "ABC"}                          # every flow moved to today
b_late = growing_annuity(OFFER_B, COST_OF_CAPITAL, INFLATION, 10)                         # as if paid at each year's end
b_now = b_late * (1 + COST_OF_CAPITAL)                                                    # each payment a year sooner
real_cc = real_rate(COST_OF_CAPITAL, INFLATION)
b_real = annuity(OFFER_B, real_cc, 10, due=True)                                          # today's dollars, real rate
b_mixed = annuity(OFFER_B, COST_OF_CAPITAL, 10, due=True)                                 # the trap: flat $2M at 9 %
monthly_cc = (1 + COST_OF_CAPITAL) ** (1 / 12) - 1
c_value = annuity(OFFER_C, monthly_cc, 120)
c_wrong = annuity(OFFER_C, COST_OF_CAPITAL / 12, 120)                                     # the trap: 9 %/12

print("1 · added up, no discounting: " + "   ".join(f"{k} {v / million:.2f}" for k, v in added.items())
      + "  ($ millions; dollars from different dates, so the sums mean nothing)")
print(f"2 · offer B at 9 %: {b_late / million:.2f} if paid at each year's end; {b_now / million:.2f} paid at the start, "
      f"as it is; from the dated cash flows {dated['B'] / million:.2f}")
print(f"3 · offer B in real terms: real rate {real_cc:.3%}, {b_real / million:.2f}: the same. Mixed up (a flat $2M at "
      f"9 %): {b_mixed / million:.2f}")
print(f"4 · offer C: monthly rate {monthly_cc:.4%}, {c_value / million:.2f}; with 0.75 % a month: {c_wrong / million:.2f}")
print(f"   at one date: A {dated['A'] / million:.2f}   B {dated['B'] / million:.2f}   C {dated['C'] / million:.2f}")
assert abs(b_late - S["B_ord"]) < 1e-6 and abs(b_now - S["B"]) < 1e-6 and abs(dated["B"] - S["B"]) < 1e-6
assert abs(b_real - S["B"]) < 1e-6 and abs(b_mixed - S["B_mixed"]) < 1e-6 and abs(real_cc - S["real"]) < 1e-12
assert abs(c_value - S["C"]) < 1e-6 and abs(dated["C"] - S["C"]) < 1e-6 and abs(c_wrong - S["C_wrong"]) < 1e-6
assert [round(v / million, 2) for v in (b_late, b_now, b_mixed, c_value, c_wrong)] == [14.41, 15.71, 13.99, 13.62, 13.42]

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.6))
fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
x = np.arange(3)
ax.bar(x - 0.19, [added[k] / million for k in "ABC"], width=0.36, color=CORAL, alpha=0.55, label="payments added up")
bars = ax.bar(x + 0.19, [dated[k] / million for k in "ABC"], width=0.36, color=SKY, label="value today, at 9 %")
bars[1].set_color(GOLD)
for i, k in enumerate("ABC"):
    ax.text(i + 0.19, dated[k] / million + 0.4, f"{dated[k] / million:.2f}", ha="center", color=GOLD if k == "B" else CHALK)
ax.set_xticks(x); ax.set_xticklabels(["A: $15M today", "B: $2M a year, rising 3 %", "C: $170,000 a month"])
ax.set_ylabel("$ millions"); ax.set_ylim(0, 26); ax.legend(loc="upper left", frameon=False)
fig.suptitle(f"At one date, offer B is worth the most: {dated['B'] / million:.2f} million, "
             f"{(dated['B'] - dated['A']) / million:.2f} more than A", x=0.08, ha="left", fontsize=15, color=GOLD)
footnote(fig, "a fictional company · our own data · computed"); plt.show()

### Your turn: tier 4

Reproduce the video's case numbers, then go further: the rate and the inflation at which offer B stops winning, what
B would be worth if it renewed forever, and your own recommendation with the number that anchors it. Use the
DataFrame `offers` and the functions above in a cell of your own (press **+** to add one), then type your answers,
in $ millions to two decimals.

In [ ]:
practice("P4.1")

In [ ]:
practice("P4.2")

In [ ]:
practice("P4.3")

In [ ]:
practice("P4.4")

In [ ]:
practice("P4.5")

In [ ]:
practice("P4.6")

In [ ]:
practice("P4.7")

In [ ]:
practice("P4.8")

### The tier 4 answers, from the DataFrame

Try the questions first. This cell works every tier 4 answer out from `offers` and the functions above and checks it
against the practice set: open it when you've finished, or when you're stuck.

In [ ]:
def value_b(r=COST_OF_CAPITAL, g=INFLATION):
    """Offer B today: ten payments growing at g, each at the start of its year, discounted at r."""
    return growing_annuity(OFFER_B, r, g, 10) * (1 + r)

mine = {}
mine["P4.1"] = [added[k] / million for k in "ABC"]
mine["P4.2"] = [b_late / million, dated["B"] / million]
mine["P4.3"] = [b_real / million, b_mixed / million]
mine["P4.4"] = [c_value / million, c_wrong / million]
even_r = solve(lambda r: value_b(r=r) - OFFER_A, 0.05, 0.20)                              # B = A at this rate
even_g = solve(lambda g: value_b(g=g) - OFFER_A, -0.05, 0.08)                             # B = A at this growth
mine["P4.5"] = [100 * even_r]
mine["P4.6"] = [100 * even_g]
mine["P4.7"] = [growing_perpetuity(OFFER_B, COST_OF_CAPITAL, INFLATION) * (1 + COST_OF_CAPITAL) / million]
mine["P4.8"] = [(dated["B"] - OFFER_A) / million]

for k, got in mine.items():
    got = [float(v) for v in got]
    assert agrees(k, got), (k, got, answers(k))
    print(f"{k}: " + " · ".join(f"{p}: {show(round(v, 4))}" for p, v in zip(Q[k]["parts"], got)))
assert abs(even_r - S["B_even_r"]) < 1e-9 and abs(even_g - S["B_even_g"]) < 1e-9
print("\nEvery tier 4 answer worked out from the DataFrame matches the practice set.")
print(f"Offer B is worth {dated['B'] / million:.2f} million, {mine['P4.8'][0]:.2f} more than A. It stays ahead while its "
      f"risk deserves less than {100 * even_r:.1f} % and inflation stays above {100 * even_g:.1f} %.")

## Sources
Every exercise, example and company here is our own (Bangalore Cold Chain is fictional). The order of topics follows
the opening chapters of a standard corporate finance course (the goal of the firm, then how to calculate present
values); the formulas are the standard ones for compound interest, perpetuities, annuities, growing streams,
effective annual rates and real rates. The spreadsheet functions named in the hints (PV, FV, PMT, RATE, NPER, NPV,
EFFECT, NOMINAL) have the same names and take their arguments in the same order in the two most widely used
spreadsheets (their official function lists, retrieved 2026-11-10).

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/cf01_present_value/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("cf01_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"]) == (_e["q"], _e["working"], _e["function"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
        _t = _e["tol"] if isinstance(_e["tol"], tuple) else (_e["tol"],) * len(_q["answer"])
        assert list(_t) == _q["tol"], _e["id"]
    assert len(_P.EX) == len(EX)
    assert all(abs(float(_P.CS[k]) - v) < 1e-9 for k, v in CASE_STATS.items()) and set(_P.CS) == set(CASE_STATS)
    assert _P.CASE["r"] == COST_OF_CAPITAL and _P.CASE["inflation"] == INFLATION == _P.CASE["B"]["g"]
    assert (_P.CASE["A"], _P.CASE["B"]["c"], _P.CASE["C"]["c"]) == (OFFER_A, OFFER_B, OFFER_C)
    assert all(abs(a - b) < 1e-6 for a, b in zip(_P.B_FLOWS, offers.loc[offers["B"] > 0, "B"]))
    assert len(_P.B_FLOWS) == (offers["B"] > 0).sum() and len(_P.C_FLOWS) == (offers["C"] > 0).sum()
    print("matches the course's practice set")